In [175]:
from bcnexus import constants as bcnexus_const
from bcnexus.clews import model_structure

In [176]:
import pandas as pd
import plotly.express as px
from pathlib import Path

# local packages

from bcnexus import utils

In [177]:
storage_algorithm="Kotzur"
nexus_scenario='Base_CNZ_noCCS'
timeslices=12
solver="gurobi"

In [178]:
from bcnexus.clews.datapackage import GetDataPackage
nexus_results_root=Path(f'results/clews/Model_{storage_algorithm}_{nexus_scenario}/{timeslices}ts_csvs_{solver}')
result_pack=GetDataPackage(nexus_results_root)
result_pack.show

└> Loaded 29 CSV files from results/clews/Model_Kotzur_Base_CNZ_noCCS/12ts_csvs_gurobi


['AccumulatedNewCapacity',
 'AnnualEmissions',
 'AnnualFixedOperatingCost',
 'AnnualTechnologyEmission',
 'AnnualTechnologyEmissionByMode',
 'AnnualVariableOperatingCost',
 'CapitalInvestment',
 'Demand',
 'DiscountedSalvageValue',
 'DiscountedTechnologyEmissionsPenalty',
 'NewCapacity',
 'NewStorageCapacity',
 'ProductionByTechnology',
 'ProductionByTechnologyAnnual',
 'RateOfActivity',
 'RateOfProductionByTechnology',
 'RateOfProductionByTechnologyByMode',
 'RateOfUseByTechnology',
 'RateOfUseByTechnologyByMode',
 'SalvageValue',
 'SalvageValueStorage',
 'StorageLevelChronoDayStart',
 'StorageLevelDayTypeFinish',
 'StorageLevelYearStart',
 'TotalAnnualTechnologyActivityByMode',
 'TotalCapacityAnnual',
 'TotalTechnologyAnnualActivity',
 'TotalTechnologyModelPeriodActivity',
 'UseByTechnology']

---

TEMP

In [179]:
# Land use

region=model_structure.Regions['REGION1'][0]

crops = {}
# DEFINE THAT THE NAMING CONVENTION FILE NEED TO LIST CROPS ALPHABETICAL

water_supply = model_structure.IrrigationTypeList

input_level = model_structure.IntensityList
mode_crop_combo=bcnexus_const.mode_of_operation

## Land use sector

In [180]:
TotalAnnualTechnologyActivityByMode_df=result_pack.get_dataframe('TotalAnnualTechnologyActivityByMode')
Landuse_df=TotalAnnualTechnologyActivityByMode_df[TotalAnnualTechnologyActivityByMode_df['TECHNOLOGY'].str.startswith('LNDAGR')].drop('REGION', axis=1)
Landuse_df['MODE_OF_OPERATION'] = Landuse_df['MODE_OF_OPERATION'].astype(int)
Landuse_df['crop_combo'] = Landuse_df['MODE_OF_OPERATION'].map(mode_crop_combo)
Landuse_df['land_use'] = Landuse_df['crop_combo']#.str[0:4]
Landuse_df.drop(['MODE_OF_OPERATION','crop_combo'], axis=1, inplace=True)

In [181]:
# Exclude rows where land_use appears among the dict values
crops_total_df = Landuse_df[~Landuse_df['land_use'].isin(model_structure.LandUseCodes.values())]

In [182]:
# crops_total_df = crops_total_df[crops_total_df['land_use'].str.startswith('CP')]
crops_total_df_pivot = crops_total_df.pivot_table(index='YEAR', 
                                            columns='land_use',
                                            values='VALUE', 
                                            aggfunc='sum').reset_index().fillna(0)
# crops_total_df_pivot = crops_total_df_pivot.reindex(sorted(crops_total_df.columns), axis=1).set_index('YEAR')

In [183]:
import pandas as pd
import plotly.express as px

def df_plot(df, y_title, p_title):
    # Melt wide columns into long form for stacked plotting
    df_melted = df.melt(
        id_vars=['YEAR'],
        var_name='Crop',
        value_name='Area'
    )

    fig = px.bar(
        df_melted,
        x='YEAR',
        y='Area',
        color='Crop',
        barmode='stack',
        labels={'Area': y_title, 'YEAR': 'Year'},
        title=p_title
    )
    fig.update_layout(
        legend_title_text='Crop',
        xaxis=dict(type='category'),
        yaxis=dict(title=y_title)
    )
    fig.show()


In [184]:
crop_cols = [c for c in crops_total_df_pivot.columns if c != 'YEAR']

# Map each crop column to its 3-letter prefix
prefix_map = {col: col[:3] for col in crop_cols}

# Group columns with same prefix and sum their values
df_grouped = crops_total_df_pivot.groupby(prefix_map,axis=1).sum()
df_grouped['YEAR'] = crops_total_df_pivot['YEAR']

crops_total_df_pivot=df_grouped

/tmp/ipykernel_3259064/2786866354.py:7: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



In [185]:
df_plot(crops_total_df_pivot,'Land area (1000 sq.km.)','Area by crop')

In [186]:
# Exclude rows where land_use appears among the dict values
land_df = Landuse_df[Landuse_df['land_use'].isin(model_structure.LandUseCodes.values())]

In [187]:
land_df = land_df.pivot_table(index='YEAR', 
                                          columns='land_use',
                                          values='VALUE', 
                                          aggfunc='sum').reset_index().fillna(0)
# land_total_df['AGR'] = 0


In [188]:
land_df['AGR'] = 0
for crop in crop_cols:
    if crop in land_df.columns:
        land_df['AGR'] += land_df[crop]
        land_df.drop(crop, axis=1, inplace=True)
# land_df = land_df.reindex(sorted(land_df.columns), axis=1).set_index('y').reset_index().rename(columns=det_col).astype('float64')

if not land_df.empty:
    df_plot(land_df,'Land area (1000 sq.km.)','Area by land cover type')

In [189]:
ProductionByTechnologyAnnual_df=result_pack.get_dataframe('ProductionByTechnologyAnnual')

In [190]:
crops_prod_df = ProductionByTechnologyAnnual_df[ProductionByTechnologyAnnual_df['FUEL'].str.startswith('CRP')]
# crops_prod_df.loc[:, 'FUEL'] = crops_prod_df['FUEL'].map(bcnexus_const.name_mapping['FUEL'])
crops_prod_df.loc[:, 'FUEL'] = crops_prod_df['FUEL'].str[3:6]

In [191]:
crops_prod_df_pivot = crops_prod_df.pivot_table(index='YEAR', 
                                          columns='FUEL',
                                          values='VALUE',
                                          aggfunc='sum').reset_index().fillna(0)
crops_prod_df_pivot = crops_prod_df_pivot.reindex(sorted(crops_prod_df_pivot.columns), axis=1).set_index('YEAR').reset_index()
# crops_prod_df['YEAR'] = years

if len(crops_prod_df_pivot.columns) > 1:
    df_plot(crops_prod_df_pivot,'Production (Million tonnes)','Crop production')

In [192]:
crops_yield_df = crops_prod_df_pivot/crops_total_df_pivot

In [195]:
crops_yield_df['YEAR']=crops_prod_df_pivot['YEAR']

In [197]:
import plotly.express as px

# Multiply all numeric columns except 'YEAR' by 10
crops_yield_df.loc[:, crops_yield_df.columns != 'YEAR'] = (
    crops_yield_df.loc[:, crops_yield_df.columns != 'YEAR'] * 10
)

# Plot only if multiple columns exist
if len(crops_yield_df.columns) > 1:
    # Melt the dataframe to long format
    df_melted = crops_yield_df.melt(
        id_vars='YEAR',
        var_name='Crop',
        value_name='Yield'
    )

    # Create interactive line+marker plot
    fig = px.line(
        df_melted,
        x='YEAR',
        y='Yield',
        color='Crop',
        markers=True,
        labels={'YEAR': 'Year', 'Yield': 'Yield (t/ha)'},
        title='Yield (tonnes per hectare)'
    )

    fig.update_traces(marker=dict(size=8))
    fig.update_layout(
        xaxis=dict(type='category'),
        yaxis=dict(title='Yield (t/ha)')
    )
    fig.show()


----

In [13]:
nexus_ts_plots={}
plot_categories = ['Climate', 'Land', 'Energy', 'Water', str(timeslices)]
nexus_plots = {category: {} for category in plot_categories}
nexus_plots[f'{timeslices}'] = nexus_ts_plots
plots = {'nexus': nexus_plots}


In [14]:
from bcnexus.vis import plot_Climate

nexus_climate_plots={}
nexus_plots['Climate'] = nexus_climate_plots

nexus_climate_plots['emission_total']=plot_Climate.get_total_annual_emission(result_pack.get_dataframe('AnnualEmissions'), nexus_scenario)
nexus_climate_plots['emission_by_source']=plot_Climate.get_emission_from_fuels(result_pack.get_dataframe('AnnualTechnologyEmission'), nexus_scenario)
nexus_climate_plots['emission_by_sector']=plot_Climate.get_emission_from_sector(result_pack.get_dataframe('AnnualTechnologyEmission'), nexus_scenario)

In [15]:
from bcnexus.vis import plot_Land
nexus_plots['Land'] = plot_Land.plot_landuse_for_clusters(result_pack.get_dataframe('RateOfProductionByTechnologyByMode'), nexus_scenario)

In [16]:
from bcnexus.vis import plot_Energy
nexus_energy_plots={}
nexus_plots['Energy'] = nexus_energy_plots

nexus_energy_plots["Nexus_sectoral_consumption"] , nexus_energy_plots["Nexus_fuel_consumption"] = plot_Energy.plot_combined_stacked_energy_consumption(result_pack.get_dataframe('UseByTechnology'), 
                                                                                                                                                       'gwh',
                                                                                                                                                       nexus_scenario)
nexus_energy_plots["Nexus_generation_from_fuels"]=plot_Energy.get_annual_generation_plot(result_pack.get_dataframe('ProductionByTechnology'), 
                                                                                  nexus_scenario)
nexus_energy_plots["Nexus_capacity_investments"]=plot_Energy.get_capacity_plot(result_pack.get_dataframe('NewCapacity'),
                                                                               nexus_scenario,
                                                                               investment=True)
nexus_energy_plots["Nexus_capacity_total"]=plot_Energy.get_capacity_plot(result_pack.get_dataframe('TotalCapacityAnnual'),nexus_scenario,investment=False)
nexus_energy_plots["Nexus_power_generation_timeslices"]=plot_Energy.get_generation_timeseries_plot(result_pack.get_dataframe('RateOfUseByTechnology'),24,nexus_scenario)
nexus_energy_plots["Nexus_power_generation_annual"]=plot_Energy.get_annual_power_generation_plot(result_pack.get_dataframe('ProductionByTechnologyAnnual'),nexus_scenario)
nexus_energy_plots["Nexus_capital_investment_power"]=plot_Energy.get_capital_investments(result_pack.get_dataframe('CapitalInvestment'),nexus_scenario)

In [17]:
nexus_energy_plots["Nexus_power_generation_annual"]

In [18]:
df = utils.add_power_tech_labels(result_pack.get_dataframe('CapitalInvestment'),'capacity')
df.power_techs_labels.unique()

array([nan, 'Wind', 'Battery Storage', 'Solar'], dtype=object)

In [19]:
nexus_energy_plots["Nexus_capital_investment_power"]

In [20]:
result_pack.get_dataframe('CapitalInvestment').TECHNOLOGY.unique()

array(['DEMAGRGWTBC1', 'DEMAGRSURBC1', 'DEMCOMBIO', 'DEMCOMELCB02',
       'DEMCOMRPP', 'DEMCOMNGS', 'DEMINDBIO', 'DEMINDDSL', 'DEMINDELCB02',
       'DEMINDNGS', 'DEMPUBGWTBC1', 'DEMPUBSURBC1', 'DEMPWRBIO',
       'DEMPWRHYD', 'DEMPWRSOL', 'DEMPWRWND', 'DEMRESBIO', 'DEMRESELCB02',
       'DEMRESRPP', 'DEMRESNGS', 'DEMTRABIO', 'DEMTRADSL', 'DEMTRAELCB02',
       'DEMTRAGSL', 'DEMTRAHFO', 'DEMTRAJFL', 'DEMTRALPG', 'DEMTRANGS',
       'IMPDSL', 'IMPGSL', 'IMPHFO', 'IMPJFL', 'IMPRPP', 'IMPLPG',
       'LNDAGRBC1C01', 'LNDAGRBC1C02', 'LNDAGRBC1C03', 'LNDAGRBC1C04',
       'LNDAGRBC1C05', 'LNDAGRBC1C06', 'LNDAGRBC1C07', 'LNDALFHRBC1',
       'LNDBARBC1', 'LNDBLTBC1', 'LNDFORBC1', 'LNDGRSBC1', 'LNDMAIHIBC1',
       'LNDOATHIBC1', 'LNDOTHHIBC1', 'LNDPEAHIBC1', 'LNDPTWHIBC1',
       'LNDRAPHIBC1', 'LNDRYEHIBC1', 'LNDWATBC1', 'LNDWHEHRBC1',
       'MINLNDBC1', 'MINNGS', 'MINPRCBC1', 'PWRTRNB01', 'PWRWNDB11',
       'RNWBIO', 'RNWHYD', 'RNWSOL', 'RNWWND', 'BATTERY_STORAGE',
       'PWRSOLB02', '

In [21]:
data=result_pack.get_dataframe('CapitalInvestment')

In [24]:
data_pwr = data.loc[data['TECHNOLOGY'].str.startswith('PWR')].copy()
data_pwr.loc[:, 'TECHNOLOGY_CODE'] = data_pwr['TECHNOLOGY'].str[:6]
data_pwr.loc[:, 'TECHNOLOGY_LABEL'] = data_pwr['TECHNOLOGY_CODE'].map(bcnexus_const.legend_labels)
data_pwr
data_pwr_grouped_data = data_pwr.groupby(['TECHNOLOGY_LABEL','TECHNOLOGY_CODE', 'YEAR', 'REGION']).sum().reset_index()


In [25]:
data_pwr_grouped_data

,TECHNOLOGY_LABEL,TECHNOLOGY_CODE,YEAR,REGION,TECHNOLOGY,VALUE,power_techs,power_techs_labels
0,Solar,PWRSOL,2030,REGION1,PWRSOLB02,154.198720,PWRSOL,Solar
1,Solar,PWRSOL,2041,REGION1,PWRSOLB04PWRSOLB07,429.461823,PWRSOLPWRSOL,SolarSolar
2,Solar,PWRSOL,2042,REGION1,PWRSOLB07,121.601990,PWRSOL,Solar
3,Solar,PWRSOL,2043,REGION1,PWRSOLB07,48.403327,PWRSOL,Solar
4,Solar,PWRSOL,2044,REGION1,PWRSOLB07,13.621037,PWRSOL,Solar
5,Solar,PWRSOL,2045,REGION1,PWRSOLB07,608.233472,PWRSOL,Solar
6,Solar,PWRSOL,2046,REGION1,PWRSOLB07,91.459473,PWRSOL,Solar
7,Transmission Grid,PWRTRN,2021,REGION1,PWRTRNB01,0.082238,0,0
8,Transmission Grid,PWRTRN,2022,REGION1,PWRTRNB01,0.087761,0,0
9,Transmission Grid,PWRTRN,2023,REGION1,PWRTRNB01,0.092518,0,0


In [26]:

import plotly.express as px

# Create a bar plot with custom legend colors
# custom_colors = bcnexus_const.custom_colors
fig = px.bar(data_pwr_grouped_data, x='YEAR', y='VALUE', color='TECHNOLOGY_LABEL', 
             labels={'VALUE': 'Mill $'}, title='Bar Plot of VALUE over Years')



fig.show()